# LeetCode 3292 — Minimum Number of Valid Strings to Form Target II bằng Python

- Link: https://leetcode.com/problems/minimum-number-of-valid-strings-to-form-target-ii/
- Độ khó: **Hard**
- Chủ đề: **"Tiền tố của tiền tố vẫn là tiền tố"** ⇒ từ mỗi vị trí nhảy được một **đoạn liên tiếp** ⇒ **Jump Game II**; tính độ dài khớp dài nhất bằng **Z-function**, **băm + nhị phân**, hoặc **Aho–Corasick**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minValidStrings(self, words: List[str], target: str) -> int:
```

## 1. Mục tiêu học

1. "Xâu hợp lệ" = tiền tố (khác rỗng) của một từ. Nếu `target[i..i+L−1]` hợp lệ thì mọi `target[i..i+L'−1]` với `L' ≤ L` cũng hợp lệ ⇒ từ vị trí `i` đi tới được **mọi** `i + 1 .. i + reach[i]`.
2. Bài toán trở thành **Jump Game II** (số bước nhảy ít nhất) — tham lam theo "tầng" BFS trong `O(n)`.
3. Tính `reach[i]` (khớp tiền tố dài nhất của một từ bắt đầu tại `i`): Z-function cho từng từ, hoặc băm + nhị phân.
4. Góc nhìn ngược: `f(e)` = số mảnh ít nhất phủ `target[:e]` **không giảm** ⇒ luôn chọn mảnh dài nhất **kết thúc** tại `e − 1` — đúng là độ sâu trạng thái **Aho–Corasick**.
5. Kiểm chứng bằng DP vét cạn.

## 2. Đề bài & ví dụ

Cho mảng `words` và chuỗi `target`. Xâu `x` **hợp lệ** nếu `x` là tiền tố của một xâu nào đó trong `words`. Trả về số xâu hợp lệ **ít nhất** ghép lại thành `target`, hoặc `-1`.

**Ví dụ 1:** `words = ["abc","aaaaa","bcdef"]`, `target = "aabcdabc"` → `3` (`"aa" + "bcd" + "abc"`).
**Ví dụ 2:** `words = ["abababab","ab"]`, `target = "ababaababa"` → `2`.
**Ví dụ 3:** `words = ["abcdef"]`, `target = "xyz"` → `-1`.

**Ràng buộc:** `1 ≤ len(words) ≤ 100`; `1 ≤ len(words[i]) ≤ 5·10^4`; `Σ len(words[i]) ≤ 10^5`; `1 ≤ len(target) ≤ 5·10^4`.

In [1]:
from typing import List
from collections import deque

examples = [(["abc", "aaaaa", "bcdef"], "aabcdabc", 3), (["abababab", "ab"], "ababaababa", 2), (["abcdef"], "xyz", -1)]
for words, t, exp in examples:
    print(words, repr(t), '-> kỳ vọng', exp)

['abc', 'aaaaa', 'bcdef'] 'aabcdabc' -> kỳ vọng 3
['abababab', 'ab'] 'ababaababa' -> kỳ vọng 2
['abcdef'] 'xyz' -> kỳ vọng -1


## 3. Kiến thức nền

### 3.1. Jump Game II (tham lam theo tầng)

Từ `i` nhảy tới tối đa `i + reach[i]`. Duyệt theo "tầng": `cur` = xa nhất đạt được với `k` bước; quét mọi `i ≤ cur` để tìm xa nhất với `k + 1` bước.

In [2]:
def jump_game(reach):
    n = len(reach)
    steps, cur, i = 0, 0, 0
    while cur < n:
        far = cur
        while i <= cur and i < n:
            far = max(far, i + reach[i]); i += 1
        if far <= cur:
            return -1
        cur = far; steps += 1
    return steps

print(jump_game([2, 3, 1, 1, 4]), jump_game([1, 0, 5]))

3 -1


### 3.2. Z-function cho độ dài khớp tiền tố

`Z(w + '#' + target)[|w| + 1 + i]` = độ dài khớp giữa `w` và `target[i:]`. Lấy max qua mọi từ ⇒ `reach[i]`. Tổng chi phí `O(Σ (|w| + n))` — với 100 từ là `≈ 5·10^6` bước.

In [3]:
def z_function(s):
    n = len(s); z = [0] * n; l = r = 0
    for i in range(1, n):
        if i < r:
            z[i] = min(r - i, z[i - l])
        while i + z[i] < n and s[z[i]] == s[i + z[i]]:
            z[i] += 1
        if i + z[i] > r:
            l, r = i, i + z[i]
    return z

def reach_by_z(words, t):
    n = len(t); reach = [0] * n
    for w in words:
        z = z_function(w + '#' + t); m = len(w)
        for i in range(n):
            if z[m + 1 + i] > reach[i]:
                reach[i] = z[m + 1 + i]
    return reach

print('reach (ví dụ 1):', reach_by_z(*examples[0][:2]))

reach (ví dụ 1): [2, 3, 3, 0, 0, 3, 2, 0]


### 3.3. Aho–Corasick

Dựng trie của mọi từ (mỗi nút = một tiền tố = một xâu hợp lệ), thêm liên kết thất bại. Khi đọc `target`, trạng thái hiện tại là **hậu tố dài nhất của phần đã đọc** mà là một nút trie ⇒ độ sâu nút = mảnh hợp lệ dài nhất **kết thúc** tại vị trí hiện tại.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `reach = [2, 1, 3, 2, 1, 3, 2, 1]` (ví dụ `reach[2] = 3` vì `"bcd"` là tiền tố của `"bcdef"`).

- Bước 1: từ 0 tới được `[1, 2]`.
- Bước 2: từ `1` tới `2`, từ `2` tới `5` ⇒ xa nhất 5.
- Bước 3: từ `3..5`, `reach[5] = 3` ⇒ tới `8 = n` ✓ ⇒ `3`.

## 5. Cách 1 — Brute Force: DP trên mọi đoạn

In [4]:
def brute(words, t):
    P = {w[:k] for w in words for k in range(1, len(w) + 1)}
    n = len(t); INF = float('inf')
    f = [INF] * (n + 1); f[0] = 0
    for e in range(1, n + 1):
        for s in range(e):
            if f[s] + 1 < f[e] and t[s:e] in P:
                f[e] = f[s] + 1
    return f[n] if f[n] < INF else -1

for words, t, exp in examples:
    print(brute(words, t), '| kỳ vọng', exp)

3 | kỳ vọng 3
2 | kỳ vọng 2
-1 | kỳ vọng -1


**Độ phức tạp:** `O(n² · L)` với tập tiền tố cỡ `Σ|w|²` — chỉ cho dữ liệu nhỏ.

## 6. Ý tưởng thuật toán / bước đột phá

Hai cách nhìn tương đương:

- **Xuôi:** tính `reach[i]` rồi Jump Game II.
- **Ngược:** `f(e)` = số mảnh ít nhất phủ `target[:e]`. `f` **không giảm** (bỏ ký tự cuối của mảnh cuối vẫn là tiền tố hợp lệ, hoặc mảnh biến mất). Vậy với mảnh cuối kết thúc tại `e − 1`, chọn mảnh **dài nhất** `d(e)` là tối ưu: `f(e) = f(e − d(e)) + 1`. `d(e)` chính là độ sâu trạng thái Aho–Corasick sau khi đọc `target[e − 1]`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Z-function cho từng từ + Jump Game II

In [5]:
def z_solution(words, t):
    return jump_game(reach_by_z(words, t))

for words, t, exp in examples:
    print(z_solution(words, t), '| kỳ vọng', exp)

3 | kỳ vọng 3
2 | kỳ vọng 2
-1 | kỳ vọng -1


### 7.2. Cách 3 — Băm mọi tiền tố + nhị phân độ dài khớp

In [6]:
def hash_solution(words, t):
    MOD, B = (1 << 61) - 1, 131
    prefixes = set()
    maxlen = 0
    for w in words:
        h = 0
        for k, c in enumerate(w, 1):
            h = (h * B + ord(c)) % MOD
            prefixes.add((k, h))
        maxlen = max(maxlen, len(w))
    n = len(t)
    pw = [1] * (n + 1); H = [0] * (n + 1)
    for i, c in enumerate(t):
        pw[i + 1] = pw[i] * B % MOD
        H[i + 1] = (H[i] * B + ord(c)) % MOD
    get = lambda l, r: (H[r] - H[l] * pw[r - l]) % MOD
    reach = [0] * n
    for i in range(n):
        lo, hi = 0, min(maxlen, n - i)
        while lo < hi:                               # "khớp được độ dài L" đơn điệu theo L
            mid = (lo + hi + 1) // 2
            if (mid, get(i, i + mid)) in prefixes:
                lo = mid
            else:
                hi = mid - 1
        reach[i] = lo
    return jump_game(reach)

for words, t, exp in examples:
    print(hash_solution(words, t), '| kỳ vọng', exp)

3 | kỳ vọng 3
2 | kỳ vọng 2
-1 | kỳ vọng -1


### 7.3. Cách 4 — Aho–Corasick + DP ngược (lời giải chính)

In [7]:
def build_ac(words):
    goto, fail, depth = [{}], [0], [0]
    for w in words:
        u = 0
        for c in w:
            nxt = goto[u].get(c)
            if nxt is None:
                nxt = len(goto); goto[u][c] = nxt
                goto.append({}); fail.append(0); depth.append(depth[u] + 1)
            u = nxt
    q = deque(goto[0].values())
    while q:
        u = q.popleft()
        for c, v in goto[u].items():
            f = fail[u]
            while f and c not in goto[f]:
                f = fail[f]
            cand = goto[f].get(c, 0)
            fail[v] = cand if cand != v else 0
            q.append(v)
    return goto, fail, depth

def ac_solution(words, t):
    goto, fail, depth = build_ac(words)
    n = len(t)
    f = [0] * (n + 1)
    u = 0
    for e in range(1, n + 1):
        c = t[e - 1]
        while u and c not in goto[u]:
            u = fail[u]
        u = goto[u].get(c, 0)
        d = depth[u]
        if d == 0:
            return -1
        f[e] = f[e - d] + 1
    return f[n]

for words, t, exp in examples:
    print(ac_solution(words, t), '| kỳ vọng', exp)

3 | kỳ vọng 3
2 | kỳ vọng 2
-1 | kỳ vọng -1


Nếu một vị trí `e` có `d = 0` thì không mảnh hợp lệ nào kết thúc tại `e − 1`, nên `f(e) = ∞`; do `f` không giảm, `f(n) = ∞` ⇒ trả `-1` ngay.

## 8. TRACE từng bước

Trace cách 4 trên ví dụ 1.

In [8]:
def trace(words, t):
    goto, fail, depth = build_ac(words)
    f = [0] * (len(t) + 1); u = 0
    for e in range(1, len(t) + 1):
        c = t[e - 1]
        while u and c not in goto[u]:
            u = fail[u]
        u = goto[u].get(c, 0)
        d = depth[u]
        f[e] = f[e - d] + 1 if d else float('inf')
        print(f"đọc {c!r} (e = {e}): mảnh dài nhất kết thúc ở đây = {t[e - d:e]!r} (d = {d}) -> f({e}) = f({e - d}) + 1 = {f[e]}")
    print('đáp án:', f[-1], '| vét cạn:', brute(words, t))

trace(*examples[0][:2])

đọc 'a' (e = 1): mảnh dài nhất kết thúc ở đây = 'a' (d = 1) -> f(1) = f(0) + 1 = 1
đọc 'a' (e = 2): mảnh dài nhất kết thúc ở đây = 'aa' (d = 2) -> f(2) = f(0) + 1 = 1
đọc 'b' (e = 3): mảnh dài nhất kết thúc ở đây = 'ab' (d = 2) -> f(3) = f(1) + 1 = 2
đọc 'c' (e = 4): mảnh dài nhất kết thúc ở đây = 'abc' (d = 3) -> f(4) = f(1) + 1 = 2
đọc 'd' (e = 5): mảnh dài nhất kết thúc ở đây = 'bcd' (d = 3) -> f(5) = f(2) + 1 = 2
đọc 'a' (e = 6): mảnh dài nhất kết thúc ở đây = 'a' (d = 1) -> f(6) = f(5) + 1 = 3
đọc 'b' (e = 7): mảnh dài nhất kết thúc ở đây = 'ab' (d = 2) -> f(7) = f(5) + 1 = 3
đọc 'c' (e = 8): mảnh dài nhất kết thúc ở đây = 'abc' (d = 3) -> f(8) = f(5) + 1 = 3
đáp án: 3 | vét cạn: 3


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def minValidStrings(self, words: List[str], target: str) -> int:
        goto = [{}]
        fail = [0]
        depth = [0]
        for w in words:
            u = 0
            for c in w:
                nxt = goto[u].get(c)
                if nxt is None:
                    nxt = len(goto)
                    goto[u][c] = nxt
                    goto.append({})
                    fail.append(0)
                    depth.append(depth[u] + 1)
                u = nxt
        q = deque(goto[0].values())
        while q:
            u = q.popleft()
            for c, v in goto[u].items():
                f = fail[u]
                while f and c not in goto[f]:
                    f = fail[f]
                cand = goto[f].get(c, 0)
                fail[v] = cand if cand != v else 0
                q.append(v)
        n = len(target)
        best = [0] * (n + 1)
        u = 0
        for e in range(1, n + 1):
            c = target[e - 1]
            while u and c not in goto[u]:
                u = fail[u]
            u = goto[u].get(c, 0)
            d = depth[u]
            if d == 0:
                return -1
            best[e] = best[e - d] + 1
        return best[n]

for words, t, exp in examples:
    got = Solution().minValidStrings(words, t)
    print(got, 'OK' if got == exp else 'SAI')

3 OK
2 OK
-1 OK


## 10. Vì sao thuật toán đúng

1. **Đóng theo tiền tố:** tiền tố của một xâu hợp lệ là xâu hợp lệ ⇒ từ `i` đi được mọi `i + L` với `1 ≤ L ≤ reach[i]` — chính là Jump Game.
2. **`f` không giảm:** lấy cách phủ tối ưu `target[:e]`; bỏ ký tự cuối khỏi mảnh cuối: nếu mảnh còn khác rỗng thì vẫn là tiền tố của cùng một từ; nếu rỗng thì bỏ mảnh ⇒ `f(e − 1) ≤ f(e)`.
3. **Mảnh dài nhất kết thúc tại `e − 1` là tối ưu:** mọi mảnh cuối hợp lệ `target[s..e−1]` có `s ≥ e − d(e)`, nên `f(e) = 1 + min_s f(s) = 1 + f(e − d(e))` (do `f` không giảm).
4. **Aho–Corasick:** trạng thái sau khi đọc `target[:e]` là nút trie sâu nhất là hậu tố của `target[:e]` (tính chất chuẩn của automaton) ⇒ độ sâu = `d(e)`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. DP mọi đoạn | `O(n² L)` | `O(Σ|w|²)` |
| 2. Z-function từng từ | `O(Σ (|w| + n))` | `O(n + max|w|)` |
| 3. Băm + nhị phân | `O(Σ|w| + n log L)` | `O(Σ|w| + n)` |
| 4. Aho–Corasick | `O(Σ|w| · σ + n)` (đã khấu trừ) | `O(Σ|w|)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'z': z_solution, 'hash': hash_solution, 'aho_corasick': ac_solution,
         'submit': lambda w, t: Solution().minValidStrings(w, t)}
edge = [(["a"], "a", 1), (["a"], "aaaa", 4), (["ab"], "ba", -1), (["abc", "bc"], "bcabc", 2),
        (["aab", "b"], "aabab", 3), (["ba", "ab"], "abab", 2)]
for words, t, exp in examples + edge:
    assert brute(words, t) == exp, (words, t, brute(words, t))
    for name, f in impls.items():
        assert f(words, t) == exp, (name, words, t)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3292)
cnt = 0
for _ in range(1200):
    words = [''.join(rng.choice('ab') for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(1, 4))]
    t = ''.join(rng.choice('ab') for _ in range(rng.randint(1, 12)))
    exp = brute(words, t)
    for name, f in impls.items():
        assert f(words, t) == exp, (name, words, t)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test.')

Khớp vét cạn trên 1200 bộ test.


### 12.3. Ca lớn (`Σ|w| = 10^5`, `|target| = 5·10^4`)

In [12]:
import time
big = {
    'một từ dài = target lặp': (['ab' * 25000], 'ab' * 25000),
    '100 từ ngẫu nhiên': ([''.join(rng.choice('ab') for _ in range(1000)) for _ in range(100)],
                          ''.join(rng.choice('ab') for _ in range(5 * 10 ** 4))),
    'chỉ "a" và "b"': (['a', 'b'], ''.join(rng.choice('ab') for _ in range(5 * 10 ** 4))),
}
for name, (words, t) in big.items():
    tt = time.time(); a = Solution().minValidStrings(words, t); t1 = time.time() - tt
    tt = time.time(); b = hash_solution(words, t); t2 = time.time() - tt
    assert a == b
    print(f"{name:26s}: {a:6d} | nộp {t1:.2f}s | băm {t2:.2f}s")
assert Solution().minValidStrings(['ab' * 25000], 'ab' * 25000) == 1
assert Solution().minValidStrings(['a', 'b'], 'ab' * 100) == 200
print('Kiểm tra độc lập: target là một từ -> 1; chỉ có từ 1 ký tự -> n mảnh.')

một từ dài = target lặp   :      1 | nộp 0.04s | băm 0.58s


100 từ ngẫu nhiên         :   6306 | nộp 0.14s | băm 0.31s
chỉ "a" và "b"            :  50000 | nộp 0.01s | băm 0.05s
Kiểm tra độc lập: target là một từ -> 1; chỉ có từ 1 ký tự -> n mảnh.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tham lam "luôn nhảy xa nhất từ vị trí hiện tại"

In [13]:
def wrong_greedy_far(words, t):
    reach = reach_by_z(words, t)
    i, steps = 0, 0
    while i < len(t):
        if reach[i] == 0:
            return -1
        i += reach[i]; steps += 1                  # SAI: không xét các điểm dừng ngắn hơn
    return steps

bad = (["a", "abab", "aa"], "aab")
print('sai :', wrong_greedy_far(*bad), '| đúng:', Solution().minValidStrings(*bad))

sai : -1 | đúng: 2


**Vì sao sai:** nhảy xa nhất có thể rơi vào vị trí "xấu" — phải xét mọi điểm trong tầng (Jump Game II) hoặc dùng DP. Với `["a", "abab", "aa"]`, `"aab"`: từ 0 nhảy xa nhất 2 (`"aa"`) rồi kẹt ở `'b'`; đúng ra phải đi `"a" + "ab"`. Ô dưới tìm tự động phản ví dụ nhỏ nhất để kiểm tra.

In [14]:
found = None
r_ = random.Random(9)
for _ in range(20000):
    words = [''.join(r_.choice('ab') for _ in range(r_.randint(1, 4))) for _ in range(r_.randint(1, 3))]
    t = ''.join(r_.choice('ab') for _ in range(r_.randint(1, 8)))
    if wrong_greedy_far(words, t) != brute(words, t):
        if found is None or len(t) < len(found[1]):
            found = (words, t)
print('phản ví dụ nhỏ nhất tìm được:', found, '-> sai:', wrong_greedy_far(*found), '| đúng:', brute(*found))

phản ví dụ nhỏ nhất tìm được: (['a', 'abab', 'aa'], 'aab') -> sai: -1 | đúng: 2


### Lỗi 2 — Chỉ coi **cả từ** là hợp lệ (quên tiền tố)

In [15]:
def wrong_whole_words(words, t):
    W = set(words); n = len(t); INF = float('inf')
    f = [INF] * (n + 1); f[0] = 0
    for e in range(1, n + 1):
        for s in range(e):
            if t[s:e] in W:                        # SAI: tiền tố của từ cũng hợp lệ
                f[e] = min(f[e], f[s] + 1)
    return f[n] if f[n] < INF else -1

print('sai :', wrong_whole_words(*examples[0][:2]), '| đúng:', Solution().minValidStrings(*examples[0][:2]))

sai : -1 | đúng: 3


**Vì sao sai:** `"aa"`, `"bcd"` chỉ là tiền tố, không phải từ đầy đủ.

### Lỗi 3 — Aho–Corasick nhưng chọn mảnh **ngắn nhất** kết thúc tại vị trí

In [16]:
def wrong_shortest(words, t):
    goto, fail, depth = build_ac(words)
    f = [0] * (len(t) + 1); u = 0
    for e in range(1, len(t) + 1):
        c = t[e - 1]
        while u and c not in goto[u]:
            u = fail[u]
        u = goto[u].get(c, 0)
        if depth[u] == 0:
            return -1
        v = u
        while depth[fail[v]] > 0:                  # SAI: đi tới hậu tố hợp lệ ngắn nhất
            v = fail[v]
        f[e] = f[e - depth[v]] + 1
    return f[-1]

print('sai :', wrong_shortest(*examples[1][:2]), '| đúng:', Solution().minValidStrings(*examples[1][:2]))

sai : 6 | đúng: 2


**Vì sao sai:** cần mảnh **dài nhất** (kết hợp với tính không giảm của `f`); mảnh ngắn làm tăng số mảnh.

### Lỗi 4 — Dùng trie nhưng **không có liên kết thất bại** (rớt thì quay về gốc)

In [17]:
def wrong_no_fail(words, t):
    goto, _, depth = build_ac(words)
    f = [0] * (len(t) + 1); u = 0
    for e in range(1, len(t) + 1):
        c = t[e - 1]
        u = goto[u][c] if c in goto[u] else goto[0].get(c, 0)   # SAI: bỏ qua các hậu tố dài hơn 1
        if depth[u] == 0:
            return -1
        f[e] = f[e - depth[u]] + 1
    return f[-1]

found4 = None
r4 = random.Random(21)
for _ in range(20000):
    words = [''.join(r4.choice('ab') for _ in range(r4.randint(1, 4))) for _ in range(r4.randint(1, 3))]
    t = ''.join(r4.choice('ab') for _ in range(r4.randint(1, 8)))
    if wrong_no_fail(words, t) != brute(words, t):
        if found4 is None or len(t) < len(found4[1]):
            found4 = (words, t)
print('phản ví dụ:', found4, '-> sai:', wrong_no_fail(*found4), '| đúng:', Solution().minValidStrings(*found4))

phản ví dụ: (['baa', 'bbb', 'b'], 'bba') -> sai: -1 | đúng: 2


**Vì sao sai:** khi không đi tiếp được trong trie, hậu tố dài nhất của phần đã đọc mà là tiền tố của một từ có thể dài hơn 1 ký tự (ví dụ trên: đọc `"bb"` theo nhánh `"bbb"`, gặp `'a'` thì nhánh rớt; hậu tố `"b"` vẫn mở và `"ba"` là tiền tố của `"baa"`, nhưng quay về gốc thì `'a'` không đi được ⇒ `-1` sai, đúng là `"b" + "ba"`). Quay về gốc làm mất thông tin đó — chính là lý do cần liên kết thất bại.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Ghép ít mảnh nhất, mảnh là **tiền tố** của từ điển" ⇒ tập lựa chọn tại mỗi vị trí là **đoạn liên tiếp** ⇒ Jump Game II.
- Khớp nhiều mẫu cùng lúc trên một văn bản ⇒ Aho–Corasick; độ sâu trạng thái = hậu tố dài nhất là tiền tố của một mẫu.
- Số từ nhỏ (≤ 100) ⇒ Z-function từng từ cũng đủ nhanh.

## 15. Biến thể của bài

1. **LC 3291 (bản I):** kích thước nhỏ hơn — DP + trie đủ.
2. Trả về **một cách ghép** tối ưu (lưu điểm cắt).
3. Mảnh là **hậu tố** của từ ⇒ đảo ngược mọi chuỗi.
4. Mỗi từ có **chi phí** riêng (mảnh từ từ `i` tốn `cost[i]`) ⇒ không còn tham lam, cần DP + Aho–Corasick duyệt chuỗi output.

In [18]:
# Biến thể 2 và 3
def one_partition(words, t):
    goto, fail, depth = build_ac(words)
    n = len(t); f = [0] * (n + 1); cut = [0] * (n + 1); u = 0
    for e in range(1, n + 1):
        c = t[e - 1]
        while u and c not in goto[u]:
            u = fail[u]
        u = goto[u].get(c, 0)
        if depth[u] == 0:
            return None
        f[e] = f[e - depth[u]] + 1; cut[e] = e - depth[u]
    parts, e = [], n
    while e:
        parts.append(t[cut[e]:e]); e = cut[e]
    return parts[::-1]

def suffix_variant(words, t):
    return Solution().minValidStrings([w[::-1] for w in words], t[::-1])

rng = random.Random(15)
for _ in range(300):
    words = [''.join(rng.choice('ab') for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 3))]
    t = ''.join(rng.choice('ab') for _ in range(rng.randint(1, 9)))
    parts = one_partition(words, t)
    exp = brute(words, t)
    if exp == -1:
        assert parts is None
    else:
        P = {w[:k] for w in words for k in range(1, len(w) + 1)}
        assert ''.join(parts) == t and len(parts) == exp and all(p in P for p in parts)
    S = {w[-k:] for w in words for k in range(1, len(w) + 1)}
    n = len(t); INF = float('inf'); g = [INF] * (n + 1); g[0] = 0
    for e in range(1, n + 1):
        for s in range(e):
            if t[s:e] in S:
                g[e] = min(g[e], g[s] + 1)
    assert suffix_variant(words, t) == (g[n] if g[n] < INF else -1)
print('Biến thể 2 (dựng cách ghép) và 3 (hậu tố) khớp vét cạn. Ví dụ:', one_partition(*examples[0][:2]))

Biến thể 2 (dựng cách ghép) và 3 (hậu tố) khớp vét cạn. Ví dụ: ['aa', 'bcd', 'abc']


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | DP mọi đoạn | `O(n² L)` | Oracle |
| 2 | Z-function + Jump Game | `O(Σ(|w| + n))` | Đơn giản |
| 3 | Băm + nhị phân | `O(n log L)` | Xác suất |
| 4 | Aho–Corasick + DP ngược | `O(Σ|w| + n)` | **Nộp bài** |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `words = ["abc"]`, `target = "abcabc"`.

**Bài 2 (tính tay):** `words = ["ab", "ba"]`, `target = "aba"`.

**Bài 3 (điền TODO):** trả về mảng `reach` bằng **trie** (duyệt từ mỗi `i` dọc trie) và so với Z.

**Bài 4 (tự viết lại):** đếm số **cách** ghép tối ưu (modulo `10^9 + 7`).

**Bài 5 (mini challenge):** biến thể 4 — chi phí theo từ.

In [19]:
def reach_by_trie_todo(words, t):
    # TODO
    raise NotImplementedError

def count_optimal_todo(words, t):
    # TODO
    raise NotImplementedError

def min_cost_todo(words, costs, t):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `"abc"` hai lần. (2) Không có mảnh dài hơn 3. (3) `2`.

**Bài 2.** (1) `"ab" + "a"` (`"a"` là tiền tố của `"ab"`). (2) Hoặc `"a" + "ba"`. (3) `2`.

**Bài 3.** (1) Dựng trie. (2) Từ mỗi `i` đi theo ký tự tới khi rớt. (3) `O(n · L)` — chậm khi từ dài, nhưng dễ hiểu.

**Bài 4.** (1) DP `(f, ways)` trên mọi đoạn (vét cạn nhỏ). (2) Nhiều cách tối ưu không nhất thiết dùng mảnh dài nhất. (3) Dùng tập tiền tố.

**Bài 5.** (1) `cost` của mảnh = min cost các từ có tiền tố đó ⇒ gán `mincost` cho mỗi nút trie. (2) DP `g[e] = min_{mảnh kết thúc tại e} g[s] + cost`. (3) Duyệt chuỗi fail từ trạng thái hiện tại (mọi hậu tố là nút trie).

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().minValidStrings(["abc"], "abcabc") == 2
assert Solution().minValidStrings(["ab", "ba"], "aba") == 2

def reach_by_trie(words, t):
    root = {}
    for w in words:
        node = root
        for c in w:
            node = node.setdefault(c, {})
    reach = []
    for i in range(len(t)):
        node, L = root, 0
        while i + L < len(t) and t[i + L] in node:
            node = node[t[i + L]]; L += 1
        reach.append(L)
    return reach

rng = random.Random(3)
for _ in range(200):
    words = [''.join(rng.choice('abc') for _ in range(rng.randint(1, 5))) for _ in range(rng.randint(1, 4))]
    t = ''.join(rng.choice('abc') for _ in range(rng.randint(1, 12)))
    assert reach_by_trie(words, t) == reach_by_z(words, t)
print('Bài 1: 2 | Bài 2: 2 | Bài 3: reach bằng trie khớp Z-function.')

Bài 1: 2 | Bài 2: 2 | Bài 3: reach bằng trie khớp Z-function.


In [21]:
# Bài 4, 5
def count_optimal(words, t, MOD=10 ** 9 + 7):
    P = {w[:k] for w in words for k in range(1, len(w) + 1)}
    n = len(t); INF = float('inf')
    f = [INF] * (n + 1); ways = [0] * (n + 1); f[0] = 0; ways[0] = 1
    for e in range(1, n + 1):
        for s in range(e):
            if f[s] < INF and t[s:e] in P:
                if f[s] + 1 < f[e]:
                    f[e], ways[e] = f[s] + 1, ways[s]
                elif f[s] + 1 == f[e]:
                    ways[e] = (ways[e] + ways[s]) % MOD
    return ways[n] if f[n] < INF else 0

def min_cost(words, costs, t):
    goto, fail, depth = build_ac(words)
    best = [float('inf')] * len(goto)
    for w, c in zip(words, costs):
        u = 0
        for ch in w:
            u = goto[u][ch]; best[u] = min(best[u], c)
    n = len(t); INF = float('inf'); g = [INF] * (n + 1); g[0] = 0; u = 0
    for e in range(1, n + 1):
        ch = t[e - 1]
        while u and ch not in goto[u]:
            u = fail[u]
        u = goto[u].get(ch, 0)
        v = u
        while v:                                   # mọi hậu tố là nút trie
            if g[e - depth[v]] + best[v] < g[e]:
                g[e] = g[e - depth[v]] + best[v]
            v = fail[v]
    return g[n] if g[n] < INF else -1

rng = random.Random(4)
for _ in range(200):
    words = [''.join(rng.choice('ab') for _ in range(rng.randint(1, 4))) for _ in range(rng.randint(1, 3))]
    costs = [rng.randint(1, 9) for _ in words]
    t = ''.join(rng.choice('ab') for _ in range(rng.randint(1, 8)))
    mc = {}
    for w, c in zip(words, costs):
        for k in range(1, len(w) + 1):
            mc[w[:k]] = min(mc.get(w[:k], 10 ** 9), c)
    n = len(t); INF = float('inf'); g = [INF] * (n + 1); g[0] = 0
    for e in range(1, n + 1):
        for s in range(e):
            if t[s:e] in mc:
                g[e] = min(g[e], g[s] + mc[t[s:e]])
    assert min_cost(words, costs, t) == (g[n] if g[n] < INF else -1)
print('Bài 5 khớp vét cạn. Bài 4 ví dụ 2:', count_optimal(*examples[1][:2]), 'cách ghép tối ưu.')

Bài 5 khớp vét cạn. Bài 4 ví dụ 2: 1 cách ghép tối ưu.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
dựng trie + fail (BFS; nút độ sâu 1 có fail = 0)
u = 0; f[0] = 0
for e in 1..n: u = chuyển(u, target[e-1]); d = depth[u]
    if d == 0: return -1
    f[e] = f[e - d] + 1
return f[n]
```

**Checklist:**

- [ ] Xâu hợp lệ gồm **mọi tiền tố** của từ?
- [ ] Dùng mảnh **dài nhất** kết thúc tại vị trí (vì `f` không giảm)?
- [ ] Liên kết thất bại không trỏ vào chính nút?
- [ ] `d = 0` ⇒ `-1` ngay?
- [ ] Nếu dùng Jump Game: xét cả tầng, không chỉ nhảy xa nhất?